In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

EMAIL_USER = os.getenv("EMAIL_USER")
EMAIL_PASS = os.getenv("EMAIL_PASS")

print("USER:", EMAIL_USER)
print("PASS:", EMAIL_PASS)
print("LEN:", len(EMAIL_PASS))

USER: patilatharv701@gmail.com
PASS: nufrzmbalntrhwvk
LEN: 16


In [3]:
import os

api_key = os.getenv("NVIDIA_API_KEY")

In [4]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
model = ChatNVIDIA(
  model="meta/llama-3.3-70b-instruct",
  api_key=api_key, 
  temperature=0.2,
  top_p=0.7,
  max_completion_tokens = 1024,
)

In [5]:
from datetime import date , timedelta
import sqlite3
import pandas as pd
from langchain_core.prompts import PromptTemplate

In [6]:
conn = sqlite3.connect('MoneyWise.db')
cursor = conn.cursor()

In [7]:
def month_end():

    today = date.today()
    tomorrow = today + timedelta(days=1)

    # if tomorrow.day == 1:
    if True:
        # cursor.execute("""
        #     SELECT *
        #     FROM transactions
        #     WHERE strftime('%m', Date) = strftime('%m', 'now')
        #     AND strftime('%Y', Date) = strftime('%Y', 'now')
        # """)

        cursor.execute("SELECT * FROM Transactions WHERE strftime('%m', Date) = '11' AND strftime('%Y', Date) = '2024'")

        month_rows = cursor.fetchall()

        if not month_rows:
            print("No transactions found")
            return

        month_df = pd.DataFrame(month_rows, columns=[
            "Id",
            "Date",
            "Title",
            "Amount",
            "Type",
            "Category",
            "Payment_method"
        ])

        month_df['Id'] = range(1, len(month_df) + 1)

        expense = month_df[
            month_df["Type"] == "Expense"
        ]["Amount"].sum()

        income = month_df[
            month_df["Type"] == "Income"
        ]["Amount"].sum()

        savings = income - expense

        print("Total Expense:", expense)
        print("Total Income:", income)
        print("Total Savings:", savings)
        display(month_df)

        month_df.to_csv("monthly_report.csv" , index = False)

    return month_df , income , expense , savings

In [8]:
month_df , income , expense , savings = month_end()

Total Expense: 12520
Total Income: 34318
Total Savings: 21798


,Id,Date,Title,Amount,Type,Category,Payment_method
0,1,2024-11-01,Design Work,34318,Income,Freelance,Bank Transfer
1,2,2024-11-30,Groceries,1330,Expense,Food,Bank Transfer
2,3,2024-11-10,Flight,4438,Expense,Transport,UPI
3,4,2024-11-06,Netflix Subscription,2875,Expense,Entertainment,Card
4,5,2024-11-06,Petrol,1305,Expense,Transport,UPI
5,6,2024-11-04,Games,1796,Expense,Entertainment,Card
6,7,2024-11-06,Gym Membership,776,Expense,Health,Cash


In [9]:
category_summary = (
    month_df[month_df["Type"] == "Expense"]
    .groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

In [10]:
prompt = f"""
You are a professional personal finance assistant.

Analyze the user's monthly financial data and generate a personalized financial report directly addressing the user.

The response must follow this exact format:

Monthly Summary:
<Write a concise overview of the user's financial activity during the month. Use phrases like "Your financial activity", "You spent", "Your savings", etc. Make the response feel personalized and user-focused.>

Key Spending Insights:
- <Write personalized insights about the user's spending habits>
- <Mention important or unusual spending patterns>
- <Mention dominant spending categories if applicable>

Financial Suggestions:
- <Give practical and personalized financial suggestions to the user>
- <Suggestions should directly relate to the user's spending behavior>
- <Keep suggestions realistic and useful>

Financial Data:
Total Income: ₹{income}
Total Expense: ₹{expense}
Total Savings: ₹{savings}

Category-wise Expenses:
{category_summary}

Recent Transactions:
{month_df.head(15).to_string(index=False)}

Instructions:
- Address the user directly using words like "you", "your", and "your spending"
- Keep the tone professional, personalized, and analytical
- Keep the response between 200-300 words
- Mention unusual or high spending categories if present
- Suggestions should be practical and personalized
- Do not use motivational quotes or emojis
- Do not sound robotic or generic
- Avoid repeating raw numbers excessively
"""

In [11]:
email_response = model.invoke(prompt)

In [12]:
from IPython.display import Markdown, display

display(Markdown(email_response.content))

Monthly Summary: Your financial activity during the month was marked by a significant income of ₹34318, with a total expense of ₹12520, resulting in substantial savings of ₹21798. You spent a considerable amount on transportation, entertainment, and food, while also allocating funds towards health expenses. Your savings account for approximately 63% of your total income, indicating a healthy saving habit.

Key Spending Insights: 
- Your spending on transportation was notably high, accounting for nearly 46% of your total expenses, with a significant portion going towards a flight and petrol.
- Entertainment expenses were also substantial, with expenses on Netflix subscription and games.
- The health category had a relatively moderate expense, primarily attributed to a gym membership.

Financial Suggestions: 
- Consider exploring cost-effective alternatives for transportation, such as carpooling or public transport, to reduce expenses in this category.
- Review your entertainment subscriptions and usage to ensure you are utilizing the services adequately, and cancel any underused subscriptions to minimize unnecessary expenses.
- Allocate a portion of your savings towards building an emergency fund or investing in a retirement plan to further secure your financial future.

In [13]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.application import MIMEApplication



In [14]:
import os
import smtplib
from email.message import EmailMessage
from dotenv import load_dotenv

load_dotenv()

EMAIL_USER = os.getenv("EMAIL_USER")
EMAIL_PASS = os.getenv("EMAIL_PASS")


def send_monthly_report(
    receiver_email,
    llm_report,
    csv_path
):

    msg = MIMEMultipart()

    msg["Subject"] = "MoneyWise Monthly Financial Report"

    msg["From"] = EMAIL_USER

    msg["To"] = receiver_email

    html_content = f"""
    <html>

        <body style="font-family: Arial;">

            <h2>MoneyWise Monthly Report</h2>

            <pre style="
                font-family: Arial;
                white-space: pre-wrap;
                line-height: 1.6;
            ">
{llm_report}
            </pre>

            <p>
                Your monthly transaction CSV file is attached.
            </p>

        </body>

    </html>
    """

    msg.attach(MIMEText(html_content, "html" , "utf-8" ))

    with open(csv_path, "rb") as f:

        csv_data = f.read()

    #     msg.attach(
    #         csv_data,
    #         maintype="text",
    #         subtype="csv",
    #         filename="monthly_transactions.csv"
    #     )
        part = MIMEApplication(csv_data, Name="monthly_transactions.csv")
        
        # Add the header so the email client knows it's an attachment
        part['Content-Disposition'] = 'attachment; filename="monthly_transactions.csv"'
        
        # Attach to the main message
        msg.attach(part)
    

    with smtplib.SMTP("smtp.gmail.com", 587) as server:

        server.starttls()

        server.login(EMAIL_USER, EMAIL_PASS)

        server.send_message(msg)

    print("Monthly report sent successfully")

In [15]:
print(EMAIL_USER)

patilatharv701@gmail.com


In [16]:
send_monthly_report(
    receiver_email="patilatharv701@gmail.com",
    llm_report=email_response.content,
    csv_path="monthly_report.csv"
)

Monthly report sent successfully


In [17]:
print(EMAIL_PASS)

nufrzmbalntrhwvk


In [18]:
SMTP_SERVER = "smtp.sendemail.com"
SMTP_PORT = 587